### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -U langchain-text-splitters langchain-community pymupdf

In [ ]:
from unsloth import FastLanguageModel
import torch
import requests
from langchain_community.document_loaders import PyMuPDFLoader
import pandas as pd
from tqdm import tqdm
import torch
from transformers import TextStreamer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.4.8 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [14]:
url = 'https://www.lobbyregister.bundestag.de/media/2c/d5/710485/SAP-Integrated-Report-2025.pdf'

loader = PyMuPDFLoader(url)
docs = loader.load()

data = pd.DataFrame()
for i in range(len(docs)):
    data.at[i, 'page'] = int(docs[i].metadata['page'])
    data.at[i, 'text'] = docs[i].page_content

In [ ]:
system_prompt = {
    'name': 'system_prompt_v2', 'text':'''
    Respond terse like smart caveman.
    Drop all fluff.
    Use short words.
    Technical substance stay.
    Minimum tokens.
'''
}

In [16]:
MAX_NEW_TOKENS = 2056
TEMPERATURE = 1
TOP_P = 0.95
TOP_K = 20

In [ ]:
data['temp_1.0'] = None
for i, ro in tqdm(data.iterrows(), total=len(data)):
    text = ro['text']

    # handling empty text
    if pd.isna(text) or not text.strip():
      continue

    messages = [{"role": "system", "content": system_prompt['text']},
                {"role": "user", "content": f"{text}"}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # Must add for generation
        enable_thinking=False,        # Enable thinking
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    generated_ids = model.generate(
    **inputs,
    max_new_tokens = MAX_NEW_TOKENS,           # Increase for longer outputs!
    temperature = TEMPERATURE,
    top_p = TOP_P,
    top_k = TOP_K,
    do_sample = True,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

    generated = tokenizer.decode(
    generated_ids[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
    )

    data.at[i, 'temp_1.0'] = generated
data.to_csv(f'SAP_caveman.csv')